Quick and dirty notebook for simulation of coherent scattering data of magnetic samples in fraunhofer far-field regime

# Import

In [ ]:
# Import general libraries
import sys, os
from os.path import join, split
from importlib import reload
from copy import deepcopy
from tqdm.auto import tqdm
import numpy as np
# scipy
import scipy
# plotting
import matplotlib.pyplot as plt
# Interactive plotting
import ipywidgets
import time
%matplotlib widget
plt.rcParams["figure.constrained_layout.use"] = True

In [ ]:
# Imports from our own codebase
from scattering_calculator.experimental_conditions import detector, light_beam
from scattering_calculator.sample_generator import pattern_generator
from scattering_calculator.sample_generator import structures
from scattering_calculator.beam_propagator import Jones_propagator
from scattering_calculator.utils import masking,physics,image_transformator
from scattering_calculator.interactive.interactive_widgets import cimshow


### EXPERIMENTAL GEOMETRY

In [ ]:
# ===================
# MATERIAL RECIPE
# ===================
recipe = "Au(1200)/SiN(200)/Co(15)/Pt(12)/Al(12)"
#"Au(1000)/SiN(200)/Ta(5)/[Pt(1)/Co(1)]x15/Pt(3)"


# ===================
# X-ray Source
# ===================
pol="CR"
x_ray_energy = 787.9  # eV
x_ray_photon_flux = 1e10  # Photons per pulse
beam_params = light_beam.beam_parameters(
    x_ray_energy, pol
)

# Params for gaussian beam
illumination_function = "gaussian"
illumination_center = (0,0)
illumination_focus_distance = 0.#in m
illumination_fwhm = 30e-6  # in m


# ==================
# DETECTOR-BEAMSTOP GEMETRY
# ==================
detector_pixel_size = 20e-6  # in m
detector_pixel_shape = (1300,1300)
detector_distance = 0.02  # in m
detector_center = (650,650)  # in px

# Optional: Define a beamstop
beamstop_radius = 0.5e-3  # in m
beamstop_distance = 0.001  # in m
beamstop_center = np.array(detector_pixel_shape) // 2  # in px


# Basic camera parameters
exp_detector = detector.detector_layout(
    pixel_size=detector_pixel_size,
    detector_shape=detector_pixel_shape,
    distance_sample_detector=detector_distance,
    detector_center=detector_center
)
exp_detector.calc_real_space_coordinates()
exp_detector.calc_q_space_coordinates(beam_params)

# Add beamstop to detector layout
beamstop = detector.beamstop(exp_detector, beamstop_distance)
beamstop.create_circle_beamstop(beamstop_center, beamstop_radius,use_real_space_coordinates=True)
bs_mask = beamstop.return_beamstop()
exp_detector.assign_beamstop(bs_mask)




#  ===================
# BASIC SAMPLE DIMENSION PARAMETERS
# ===================

## This is the resolution we will have thanks to the detector
res_from_detector=(np.pi/np.maximum(np.amax(np.abs(exp_detector.detqx)),np.amax(np.abs(exp_detector.detqy))))
FOV_from_detector=detector_pixel_shape[0]*res_from_detector

sample_shape = np.array([0, int(2*detector_pixel_shape[0]), int(2*detector_pixel_shape[0])])  # in pixels
real_space_pixel_size = res_from_detector/4  # in m


#  ===================
# HOLOGRAPHY MASK DESIGN
# ===================

apertures_radius=[600e-9,16e-9,40e-9]
apertures_type=["OH", "RH", "RH"]
apertures_centers=[(0,0),(1.4e-6, -1.4e-6),(1.4e-6,1.4e-6)]
apertures_sigma=[10e-9,4e-9,7e-9]




apertures_radius=[60e-9,6e-9,4e-9]
apertures_type=["OH", "RH", "RH"]
apertures_centers=[(0,0),(0.2e-6, -0.2e-6),(0.15e-6,0.15e-6)]
apertures_sigma=[1e-9,0.1e-9,0.1e-9]


### SAMPLE STACK STRUCTURE AND OPTICAL PROPERTIES

In [ ]:

reload(structures)


# define stack
stack = structures.parse_recipe(
    recipe,
    sample_name="sample_A",
    comments=["test multilayer"],
)

#pulls material refractive indexes
material_params = structures.material_params(materials=set([element.material for element in stack.layers]), x_ray_energy=x_ray_energy)


sample = structures.Structure(
    name="Test Structure",
    material_params=material_params,
    sample_shape=sample_shape,
    real_space_pixel_size=real_space_pixel_size
)

for layer in stack.layers:
    sample.add_layer(layer.material, thickness=layer.thickness)



# SAMPLE 

### - magnetic domains

In [ ]:

reload(pattern_generator)

%time
# Skyrmion and Screening diameter
skyrmion_radius = 3e-9  # m
screening_radius = (
    1.5 * skyrmion_radius
)  # m, either only even or odd, otherwise script will fail
skyrmion_smoothing = 1

# Number of skyrmions
# If this number is too high, the script may take forever ...
# (brute force algorithm)
number_of_skyrmions = 600000
max_nr_iteration = 20000  # 10 * sample["no_skyr"]

# Create Pattern
skyrmion_pattern, coordinates = pattern_generator.create_skyrmion_pattern(
    sample.sample_shape[1:],
    skyrmion_radius/sample.real_space_pixel_size,
    screening_radius/sample.real_space_pixel_size,
    number_of_skyrmions,
    max_nr_iteration,
    sigma=skyrmion_smoothing,
    real_space_pixel_size = sample.real_space_pixel_size,
    plot=True,
)

sample.magnetization=pattern_generator.map_magnetization_to_3d(0*skyrmion_pattern,np.sqrt(1-np.abs(skyrmion_pattern)**2),skyrmion_pattern,nr_repeats=sample_shape[0])



# - holography mask

In [ ]:
reload(structures)
reload(masking)


front_aperture = structures.Apertures3D(sample_shape, real_space_pixel_size,
    layer_thicknesses=sample.layer_thicknesses)

for hole in range(len(apertures_type)):
    if apertures_type[hole] =="RH":
        depth=np.sum(sample.layer_thicknesses),
    else:
        depth=np.sum(np.array((sample.layer_thicknesses[:sample.layer_names.index("SiN")])))
    print(depth)

    front_aperture.create_circle_aperture(
        center=apertures_centers[hole],
        depth=depth,
        radius=apertures_radius[hole],
        use_real_space_coordinates=True,
        sigma=apertures_sigma[hole]
    )



sample.mask=front_aperture.aperture_design

front_aperture.visualize_aperture()


### HOLOGRAM COMPUTATION

In [ ]:


reload(Jones_propagator)
reload(light_beam)
reload(detector)
reload(structures)

time0=time.time()
# compute dielectric tensor
sample.calculate_final_dielectric_tensor()
print("dielectric tensor   - %0.1f s"%(time.time()-time0))
time0=time.time()

holos=[None,None]
ii=0

for beam_params.pol in ["CR", "CL"]:
    illumination = light_beam.illumination(beam_params, sample_shape[1:], real_space_pixel_size)

    # Comment: Check gauss_beam function for different focus distances
    illumination.gauss_beam(
        illumination_center,
        illumination_focus_distance,
        illumination_fwhm,
    )
    illumination.get_illumination_jones()


    illumination_wavefield = illumination.return_illumination()
    extent_illumination_real = illumination.get_illumination_extent_real_space()
    #illumination.visualize_illumination()


    print("illumination wave   - %0.1f s"%(time.time()-time0))
    time0=time.time()

    ### Do light propagation

    reload(Jones_propagator)
    reload(detector)

    wavefront = Jones_propagator.wavefronts(beam_parameters=beam_params,
                                            eps_stack=sample.final_dielectric_tensor,
                                            layer_thicknesses=sample.layer_thicknesses,
                                            real_space_pixel_size=sample.real_space_pixel_size,
                                            E_in=illumination.illumination_jones,
                                            propagate=False)


    print("jones propagation   - %0.1f s"%(time.time()-time0))
    time0=time.time()

    hologram_exp = detector.detector_hologram( exp_detector, wavefront.hologram, beam_params,sample.real_space_pixel_size, beamstop)
    hologram_exp.gnomonic_projection()
    hologram_exp.add_noise()

    print("gnomonic projection - %0.1f s"%(time.time()-time0))
    time0=time.time()

    holos[ii]=hologram_exp.hologram_exp.copy()
    ii+=1

In [ ]:
fth=Jones_propagator.reconstruct((holos[0]-holos[1])*scipy.ndimage.gaussian_filter(1.*scipy.ndimage.binary_erosion(1-beamstop.beamstop, iterations=25,border_value=1), sigma=10))
fth2=np.abs(fth)
fth2[fth.shape[0]//2:,:]=np.imag(fth)[fth.shape[0]//2:,:]
cimshow(fth2, cmap="gray")

In [ ]:
plt.close("all")


fig,ax=plt.subplots(2,7, figsize=(14,4))

roi=np.s_[900:1150,120:400]
roi2=np.s_[1:900,:1200]
roi3=np.s_[630:670,1135:1170]


fth=Jones_propagator.reconstruct((holos[0]+holos[1]))

ax[0,0].imshow(np.log10(holos[0]+holos[1]))

ax[0,1].imshow((np.real(fth)[roi]))
ax[0,2].imshow((np.imag(fth)[roi]))

ax[0,3].imshow((np.real(fth)[roi2]))
ax[0,4].imshow((np.imag(fth)[roi2]))

ax[0,5].imshow((np.real(fth)[roi3]))
ax[0,6].imshow((np.imag(fth)[roi3]))

fth=Jones_propagator.reconstruct(holos[0]-holos[1])

ax[1,0].imshow(np.log10(np.abs(holos[0]-holos[1])))

ax[1,1].imshow((np.real(fth)[roi]))
ax[1,2].imshow((np.imag(fth)[roi]))

ax[1,3].imshow((np.real(fth)[roi2]))
ax[1,4].imshow((np.imag(fth)[roi2]))

ax[1,5].imshow((np.real(fth)[roi3]))
ax[1,6].imshow((np.imag(fth)[roi3]))


In [ ]:
fth=Jones_propagator.reconstruct(holos[0]-holos[1])
fth2=np.real(fth)
fth2[fth.shape[0]//2:,:]=np.imag(fth)[fth.shape[0]//2:,:]
cimshow(fth2, cmap="gray")

In [ ]:

cimshow(np.imag(Jones_propagator.reconstruct(hologram_exp.hologram_exp)), cmap="gray")

In [ ]:
plt.close("all")


fig,ax=plt.subplots(2,7, figsize=(14,4))

roi=np.s_[270:460,300:490]
roi2=np.s_[280:460,820:990]
roi3=np.s_[630:670,1135:1170]


fth=Jones_propagator.reconstruct((holos[0]+holos[1]))

ax[0,0].imshow(np.log10(holos[0]+holos[1]))

ax[0,1].imshow((np.real(fth)[roi]))
ax[0,2].imshow((np.imag(fth)[roi]))

ax[0,3].imshow((np.real(fth)[roi2]))
ax[0,4].imshow((np.imag(fth)[roi2]))

ax[0,5].imshow((np.real(fth)[roi3]))
ax[0,6].imshow((np.imag(fth)[roi3]))

fth=Jones_propagator.reconstruct(holos[0]-holos[1])

ax[1,0].imshow(np.log10(np.abs(holos[0]-holos[1])))

ax[1,1].imshow((np.real(fth)[roi]))
ax[1,2].imshow((np.imag(fth)[roi]))

ax[1,3].imshow((np.real(fth)[roi2]))
ax[1,4].imshow((np.imag(fth)[roi2]))

ax[1,5].imshow((np.real(fth)[roi3]))
ax[1,6].imshow((np.imag(fth)[roi3]))
